# 06 — GPU RGB feature extraction (R3D-18, full sequence-video coverage)

**Corrected worker-interpreter version.** This notebook extracts one 512-dimensional pretrained Kinetics-400 R3D-18 descriptor for each source video represented in the full tracked sequence dataset from Notebook 04.

- Uses the existing group-aware train/validation/test assignments.
- Does not install, uninstall, or upgrade packages.
- Explicitly launches the worker with the existing Anaconda interpreter at `C:\Users\admin\anaconda3\python.exe`, because the previous attempt launched with the Jupyter kernel's Python 3.13 interpreter, which did not have OpenCV (`cv2`).
- Preflights OpenCV, PyTorch, TorchVision, R3D-18 imports and CUDA before launching.
- Runs one detached worker, loads R3D-18 once on `cuda:0`, and processes videos sequentially.
- Saves each video's feature atomically, writes a per-video JSON result, and updates progress state.
- Reuses verified feature files and skips completed videos when restarted.
- Writes to `data/rgb_features_r3d18_v2`, the directory expected by Notebook 07.

**Important:** Run only after Notebook 05 has finished using the GPU. The first run may download official torchvision R3D-18 Kinetics-400 weights if they are not cached. If weights cannot be obtained, the worker records the error and does not substitute random weights.


In [11]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")

# IMPORTANT: The Jupyter kernel in the previous attempt used Python 3.13,
# which did not have cv2 installed. Launch the detached worker explicitly
# with the already-existing Anaconda Python that passed the earlier package
# and YOLO/CUDA diagnostics. This does not install or change any packages.
WORKER_PYTHON = Path(r"C:\Users\admin\anaconda3\python.exe")
assert WORKER_PYTHON.is_file(), f"Existing Anaconda Python not found: {WORKER_PYTHON}"

preflight_code = (
    "import sys, cv2, torch, torchvision; "
    "from torchvision.models.video import r3d_18, R3D_18_Weights; "
    "print('PYTHON=' + sys.executable); "
    "print('CV2=' + cv2.__version__); "
    "print('TORCH=' + torch.__version__); "
    "print('TORCHVISION=' + torchvision.__version__); "
    "print('CUDA=' + str(torch.cuda.is_available())); "
    "print('GPU=' + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UNAVAILABLE'))"
)
preflight = subprocess.run(
    [str(WORKER_PYTHON), "-c", preflight_code],
    capture_output=True, text=True, errors="replace", timeout=120
)
print("RGB worker Python preflight:")
print(preflight.stdout)
if preflight.stderr.strip():
    print("Preflight stderr:", preflight.stderr[-2000:])
if preflight.returncode != 0:
    raise RuntimeError(
        "The configured Anaconda Python cannot import the RGB worker dependencies. "
        "Do not launch Cell 3. Review the preflight output above."
    )
if "CUDA=True" not in preflight.stdout:
    raise RuntimeError(
        "CUDA is not available in the configured Anaconda Python. "
        "RGB GPU extraction was not launched."
    )
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"
PTR_PATH = NOTEBOOK2_DIR / "data" / "CURRENT_TRACKED_SEQUENCE_DIR.txt"
assert PTR_PATH.is_file(), "Notebook 04 dataset pointer not found. Run corrected Notebook 04 first."
SEQUENCE_DIR = Path(PTR_PATH.read_text(encoding="utf-8").strip())
assert SEQUENCE_DIR.is_dir(), f"Sequence directory not found: {SEQUENCE_DIR}"
META_PATH = SEQUENCE_DIR / "dataset_metadata.json"
assert META_PATH.is_file(), "dataset_metadata.json is missing."
META = json.loads(META_PATH.read_text(encoding="utf-8"))
assert META.get("is_full_dataset", False), "RGB fusion requires the full tracked sequence dataset, not pilot sequences."

SPLIT_PATH = PROJECT_DIR / "data" / "processed" / "sequences_group_split_v1" / "video_split_manifest.csv"
assert SPLIT_PATH.is_file(), f"Original group-split manifest not found: {SPLIT_PATH}"
manifest = pd.read_csv(SPLIT_PATH)
required = {"video_path", "label", "split", "video_id"}
missing_columns = required.difference(manifest.columns)
assert not missing_columns, f"Manifest missing columns: {sorted(missing_columns)}"
manifest["video_id"] = pd.to_numeric(manifest["video_id"], errors="raise").astype("int64")
manifest["video_path"] = manifest["video_path"].astype(str)
manifest["label"] = manifest["label"].astype(str)
manifest["split"] = manifest["split"].astype(str).str.lower()
assert set(manifest["split"].unique()).issubset({"train", "val", "test"}), "Unexpected split values."
assert not manifest["video_id"].duplicated().any(), "video_id is not unique in the source manifest."

# Select only unique source videos represented in the full sequence arrays.
sequence_video_ids = set()
for split_name in ("train", "val", "test"):
    videos_path = SEQUENCE_DIR / f"videos_{split_name}.npy"
    y_path = SEQUENCE_DIR / f"y_{split_name}.npy"
    assert videos_path.is_file() and y_path.is_file(), f"Missing {split_name} sequence arrays."
    videos = np.load(videos_path, mmap_mode="r", allow_pickle=False)
    labels = np.load(y_path, mmap_mode="r", allow_pickle=False)
    assert len(videos) == len(labels), f"{split_name}: videos/y length mismatch."
    sequence_video_ids.update(int(v) for v in np.unique(videos))
    del videos, labels
assert sequence_video_ids, "No source video IDs were found in the full sequence arrays."
manifest_ids = set(manifest["video_id"].astype(int))
missing_ids = sorted(sequence_video_ids - manifest_ids)
assert not missing_ids, f"{len(missing_ids)} sequence video IDs are absent from source manifest. Example: {missing_ids[:10]}"
selected = manifest[manifest["video_id"].isin(sequence_video_ids)].copy()
selected = selected.sort_values(["split", "label", "video_id"]).reset_index(drop=True)
assert set(selected["video_id"].astype(int)) == sequence_video_ids

# Keep this path identical to the RGB_DIR expected by Notebook 07.
RGB_DIR = NOTEBOOK2_DIR / "data" / "rgb_features_r3d18_v2"
LOG_DIR = NOTEBOOK2_DIR / "logs" / "rgb_features_r3d18_v2"
SRC_DIR = NOTEBOOK2_DIR / "src"
RGB_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_CSV = LOG_DIR / "selected_sequence_source_videos.csv"
selected.to_csv(SELECTED_CSV, index=False)

NUM_FRAMES = 16
DEVICE = "cuda:0"  # Deliberately requires CUDA; no silent CPU fallback.
WORKER = SRC_DIR / "r3d18_feature_worker_v3.py"
CONFIG_PATH = LOG_DIR / "worker_config.json"
STATE_PATH = LOG_DIR / "progress_state.json"
PID_PATH = LOG_DIR / "worker_pid.txt"
REPORT_PATH = RGB_DIR / "rgb_feature_extraction_report.csv"
PER_VIDEO_DIR = LOG_DIR / "per_video_results"
PER_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("FULL RGB FEATURE DATASET CONFIGURED")
print("Sequence dataset:", SEQUENCE_DIR)
print("Unique source videos needed by sequences:", len(selected))
print("Video counts by split:")
print(selected.groupby("split").size().to_string())
print("Classes represented:", selected["label"].nunique())
print("RGB feature output (Notebook 07 input):", RGB_DIR)
print("Device requested:", DEVICE)
print("Selected-video manifest:", SELECTED_CSV)


RGB worker Python preflight:
PYTHON=C:\Users\admin\anaconda3\python.exe
CV2=4.13.0
TORCH=2.7.1+cu118
TORCHVISION=0.22.1+cu118
CUDA=True
GPU=NVIDIA RTX A4000

FULL RGB FEATURE DATASET CONFIGURED
Sequence dataset: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\sequences_tracked_group_split_iou_v3_full_v1\20261031_035536
Unique source videos needed by sequences: 9654
Video counts by split:
split
test     1943
train    5755
val      1956
Classes represented: 100
RGB feature output (Notebook 07 input): C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\rgb_features_r3d18_v2
Device requested: cuda:0
Selected-video manifest: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\rgb_features_r3d18_v2\selected_sequence_source_videos.csv


In [12]:
# Write the complete worker program used by the detached process.
WORKER_SOURCE = '# Worker starts in a fresh process. Preload PyTorch\'s bundled OpenMP runtime\n# before importing torch/cv2/torchvision. No package or system files are changed.\nimport os, sys, ctypes, json, time, hashlib, traceback, gc\nfrom pathlib import Path\n\nTORCH_LIB = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"\n_dll_handles = []\n_openmp_handle = None\nif os.name == "nt":\n    torch_iomp = TORCH_LIB / "libiomp5md.dll"\n    if torch_iomp.is_file():\n        try:\n            _openmp_handle = ctypes.WinDLL(str(torch_iomp))\n            print("OPENMP_PRELOADED_FROM", torch_iomp, flush=True)\n        except OSError as exc:\n            print("OPENMP_PRELOAD_WARNING", repr(exc), flush=True)\nif TORCH_LIB.is_dir():\n    os.environ["PATH"] = str(TORCH_LIB) + os.pathsep + os.environ.get("PATH", "")\n    if hasattr(os, "add_dll_directory"):\n        try:\n            _dll_handles.append(os.add_dll_directory(str(TORCH_LIB)))\n        except OSError:\n            pass\nfor name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):\n    os.environ[name] = "1"\nos.environ["PYTHONUNBUFFERED"] = "1"\n\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport torch\nfrom torchvision.models.video import r3d_18, R3D_18_Weights\n\n\ndef atomic_json(path, obj):\n    path = Path(path)\n    tmp = path.with_name(path.name + f".{os.getpid()}.tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(str(tmp), str(path))\n\n\ndef output_for(row, out_dir):\n    key = str(Path(row["video_path"]).resolve()).casefold()\n    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]\n    return Path(out_dir) / f"video_{int(row[\'video_id\']):06d}_{digest}.npz"\n\n\ndef validate_existing(path, row):\n    try:\n        with np.load(path, allow_pickle=False) as d:\n            return (\n                d["feature"].shape == (512,)\n                and int(d["video_id"]) == int(row["video_id"])\n                and str(d["video_path"].item()) == str(row["video_path"])\n                and str(d["split"].item()) == str(row["split"])\n                and np.isfinite(d["feature"]).all()\n            )\n    except Exception:\n        return False\n\n\ndef resize_center_crop(rgb, short_side=128, crop=112):\n    h, w = rgb.shape[:2]\n    if h < 1 or w < 1:\n        raise ValueError("Invalid decoded frame dimensions")\n    scale = float(short_side) / float(min(h, w))\n    nw = max(crop, int(round(w * scale)))\n    nh = max(crop, int(round(h * scale)))\n    resized = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    x0 = max(0, (nw - crop) // 2)\n    y0 = max(0, (nh - crop) // 2)\n    return resized[y0:y0 + crop, x0:x0 + crop]\n\n\ndef extract_clip(video_path, num_frames):\n    cap = cv2.VideoCapture(str(video_path))\n    if not cap.isOpened():\n        raise RuntimeError(f"OpenCV could not open video: {video_path}")\n    try:\n        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\n        if frame_count < 1:\n            raise RuntimeError("Video reports zero frames")\n        indices = np.linspace(0, max(frame_count - 1, 0), int(num_frames)).round().astype(np.int64)\n        frames = []\n        for frame_idx in indices:\n            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))\n            ok, bgr = cap.read()\n            if not ok:\n                continue\n            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n            frames.append(resize_center_crop(rgb))\n        if len(frames) < max(8, int(num_frames) // 2):\n            raise RuntimeError(f"Too few readable sampled frames: {len(frames)}")\n        return np.asarray(frames, dtype=np.uint8), frame_count\n    finally:\n        cap.release()\n\n\ndef main(cfg_path):\n    cfg = json.loads(Path(cfg_path).read_text(encoding="utf-8"))\n    out_dir = Path(cfg["rgb_dir"])\n    log_dir = Path(cfg["log_dir"])\n    per_video_dir = Path(cfg["per_video_dir"])\n    out_dir.mkdir(parents=True, exist_ok=True)\n    log_dir.mkdir(parents=True, exist_ok=True)\n    per_video_dir.mkdir(parents=True, exist_ok=True)\n    selected = pd.read_csv(cfg["selected_csv"])\n    total = len(selected)\n    state_path = Path(cfg["state_path"])\n    start = time.time()\n    success = skipped = failed = 0\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is not available in this Python environment. GPU RGB extraction was not started.")\n    if torch.cuda.device_count() < 1:\n        raise RuntimeError("No CUDA device was detected.")\n    device = torch.device(cfg.get("device", "cuda:0"))\n    torch.set_num_threads(1)\n    try:\n        torch.set_num_interop_threads(1)\n    except RuntimeError:\n        pass\n    cv2.setNumThreads(1)\n    print("RGB_WORKER_START", "videos", total, "device", device, flush=True)\n    print("GPU", torch.cuda.get_device_name(device), flush=True)\n    print("Loading pretrained R3D-18 Kinetics-400 weights...", flush=True)\n    weights = R3D_18_Weights.DEFAULT\n    model = r3d_18(weights=weights)\n    model.fc = torch.nn.Identity()\n    model = model.to(device).eval()\n    print("R3D18_READY", "device", device, "feature_dim", 512, flush=True)\n\n    atomic_json(state_path, {"status": "running", "device": str(device), "gpu_name": torch.cuda.get_device_name(device),\n                             "total_videos": total, "completed": 0, "success": 0, "skipped_existing": 0,\n                             "failed": 0, "current_video": None, "started_at": time.strftime("%Y-%m-%d %H:%M:%S")})\n\n    for pos, row in enumerate(selected.to_dict("records"), 1):\n        video_path = Path(str(row["video_path"]))\n        out_path = output_for(row, out_dir)\n        result_path = per_video_dir / f"result_{int(row[\'video_id\']):06d}.json"\n        common = {"video_id": int(row["video_id"]), "video_path": str(video_path),\n                  "label": str(row["label"]), "split": str(row["split"]), "output_path": str(out_path)}\n        try:\n            if out_path.is_file() and validate_existing(out_path, row):\n                skipped += 1\n                obj = {**common, "status": "skipped_existing", "feature_dim": 512, "device": str(device)}\n            else:\n                if not video_path.is_file():\n                    raise FileNotFoundError(f"Source video not found: {video_path}")\n                frames, frame_count = extract_clip(video_path, int(cfg["num_frames"]))\n                # [T,H,W,C] -> [1,C,T,H,W], matching torchvision R3D-18 input.\n                x = torch.from_numpy(frames).to(device=device, dtype=torch.float32)\n                x = x.permute(3, 0, 1, 2).unsqueeze(0).div_(255.0)\n                mean = torch.tensor([0.43216, 0.394666, 0.37645], device=device).view(1, 3, 1, 1, 1)\n                std = torch.tensor([0.22803, 0.22145, 0.216989], device=device).view(1, 3, 1, 1, 1)\n                x = (x - mean) / std\n                with torch.inference_mode():\n                    feat = model(x).detach().float().cpu().numpy().reshape(-1).astype(np.float32)\n                if feat.shape != (512,) or not np.isfinite(feat).all():\n                    raise RuntimeError(f"Invalid R3D feature output: shape={feat.shape}")\n                tmp = out_path.with_name(out_path.stem + f".{os.getpid()}.tmp.npz")\n                np.savez_compressed(tmp, feature=feat,\n                                    video_path=np.asarray(str(video_path)),\n                                    label=np.asarray(str(row["label"])),\n                                    video_id=np.asarray(int(row["video_id"]), dtype=np.int64),\n                                    split=np.asarray(str(row["split"])),\n                                    sampled_frames=np.asarray(len(frames), dtype=np.int32),\n                                    decoded_frames=np.asarray(frame_count, dtype=np.int32),\n                                    backbone=np.asarray("torchvision_r3d_18_kinetics400"),\n                                    device=np.asarray(str(device)))\n                os.replace(str(tmp), str(out_path))\n                success += 1\n                obj = {**common, "status": "success", "feature_dim": 512,\n                       "sampled_frames": int(len(frames)), "decoded_frames": int(frame_count), "device": str(device)}\n                del x, frames, feat\n                if device.type == "cuda":\n                    torch.cuda.empty_cache()\n                gc.collect()\n            atomic_json(result_path, obj)\n        except Exception as exc:\n            failed += 1\n            obj = {**common, "status": "failed", "error": repr(exc), "device": str(device)}\n            atomic_json(result_path, obj)\n            print("VIDEO_FAILED", json.dumps(obj), flush=True)\n\n        completed = success + skipped + failed\n        state = {"status": "running", "device": str(device), "gpu_name": torch.cuda.get_device_name(device),\n                 "total_videos": total, "completed": completed, "success": success,\n                 "skipped_existing": skipped, "failed": failed,\n                 "current_video": video_path.name, "current_label": str(row["label"]),\n                 "elapsed_seconds": round(time.time() - start, 1),\n                 "last_update": time.strftime("%Y-%m-%d %H:%M:%S")}\n        atomic_json(state_path, state)\n        if completed % 25 == 0 or completed == total:\n            print("RGB_PROGRESS", json.dumps(state), flush=True)\n\n    rows = []\n    for p in sorted(per_video_dir.glob("result_*.json")):\n        try:\n            rows.append(json.loads(p.read_text(encoding="utf-8")))\n        except Exception:\n            pass\n    if rows:\n        pd.DataFrame(rows).sort_values("video_id").to_csv(Path(cfg["report_path"]), index=False)\n    final_status = "completed_with_errors" if failed else "completed"\n    final = {"status": final_status, "device": str(device), "gpu_name": torch.cuda.get_device_name(device),\n             "total_videos": total, "completed": success + skipped + failed, "success": success,\n             "skipped_existing": skipped, "failed": failed,\n             "elapsed_seconds": round(time.time() - start, 1),\n             "report_path": str(cfg["report_path"])}\n    atomic_json(state_path, final)\n    print("RGB_FEATURE_EXTRACTION_FINISHED", json.dumps(final), flush=True)\n\n\nif __name__ == "__main__":\n    try:\n        main(sys.argv[1])\n    except BaseException as exc:\n        cfg = {}\n        try:\n            cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))\n        except Exception:\n            pass\n        state_path = Path(cfg["state_path"]) if cfg.get("state_path") else None\n        error = {"status": "worker_failed", "error": repr(exc), "traceback": traceback.format_exc(),\n                 "failed_at": time.strftime("%Y-%m-%d %H:%M:%S")}\n        print("RGB_WORKER_FATAL", json.dumps(error), flush=True)\n        if state_path:\n            try:\n                atomic_json(state_path, error)\n            except Exception:\n                pass\n        traceback.print_exc()\n        sys.exit(2)\n'
WORKER.write_text(WORKER_SOURCE, encoding="utf-8")

CONFIG = {
    "selected_csv": str(SELECTED_CSV),
    "rgb_dir": str(RGB_DIR),
    "log_dir": str(LOG_DIR),
    "per_video_dir": str(PER_VIDEO_DIR),
    "state_path": str(STATE_PATH),
    "report_path": str(REPORT_PATH),
    "num_frames": NUM_FRAMES,
    "device": DEVICE,
}
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Worker script saved:", WORKER)
print("Configuration saved:", CONFIG_PATH)
print("R3D-18 will load once and process selected videos sequentially.")


Worker script saved: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\src\r3d18_feature_worker_v3.py
Configuration saved: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\rgb_features_r3d18_v2\worker_config.json
R3D-18 will load once and process selected videos sequentially.


In [13]:
# Launch one detached GPU worker. Do not launch a duplicate if the recorded PID is alive.
import time

def is_pid_running(pid):
    if os.name == "nt":
        check = subprocess.run(["tasklist", "/FI", f"PID eq {int(pid)}"],
                               capture_output=True, text=True, errors="replace")
        return str(int(pid)) in (check.stdout or "")
    try:
        os.kill(int(pid), 0)
        return True
    except OSError:
        return False

if PID_PATH.is_file():
    try:
        old_pid = int(PID_PATH.read_text(encoding="utf-8").strip())
    except Exception:
        old_pid = -1
    if old_pid > 0 and is_pid_running(old_pid):
        raise RuntimeError(f"RGB worker PID {old_pid} is already running. Do not launch another worker; use the monitor cell.")

env = os.environ.copy()
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    env[key] = "1"
# Use the torch DLL directory belonging to WORKER_PYTHON (Anaconda),
# not the Jupyter kernel's possibly different Python 3.13 environment.
worker_prefix = WORKER_PYTHON.parent
worker_torch_lib = worker_prefix / "Lib" / "site-packages" / "torch" / "lib"
if worker_torch_lib.is_dir():
    env["PATH"] = str(worker_torch_lib) + os.pathsep + env.get("PATH", "")
env["PYTHONUNBUFFERED"] = "1"
stdout_path = LOG_DIR / "rgb_worker_stdout.log"
stderr_path = LOG_DIR / "rgb_worker_stderr.log"
# Archive logs/state from the failed previous attempt, without deleting them.
attempt_stamp = time.strftime("%Y%m%d_%H%M%S")
for old_path in (stdout_path, stderr_path, STATE_PATH):
    if old_path.is_file():
        archived = old_path.with_name(old_path.stem + "_previous_" + attempt_stamp + old_path.suffix)
        old_path.replace(archived)
creationflags = 0
if os.name == "nt":
    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS
stdout_file = open(stdout_path, "a", encoding="utf-8", buffering=1)
stderr_file = open(stderr_path, "a", encoding="utf-8", buffering=1)
try:
    proc = subprocess.Popen(
        [str(WORKER_PYTHON), str(WORKER), str(CONFIG_PATH)],
        cwd=str(PROJECT_DIR), env=env,
        stdout=stdout_file, stderr=stderr_file,
        creationflags=creationflags,
        close_fds=True,
    )
finally:
    stdout_file.close()
    stderr_file.close()
PID_PATH.write_text(str(proc.pid), encoding="utf-8")
print("RGB worker launched. PID:", proc.pid)
print("Worker Python:", WORKER_PYTHON)
print("Device requested:", DEVICE)
print("Stdout:", stdout_path)
print("Stderr:", stderr_path)
print("Use the monitor cell for updates. Do not launch a second worker while this PID is alive.")


RGB worker launched. PID: 668
Worker Python: C:\Users\admin\anaconda3\python.exe
Device requested: cuda:0
Stdout: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\rgb_features_r3d18_v2\rgb_worker_stdout.log
Stderr: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\rgb_features_r3d18_v2\rgb_worker_stderr.log
Use the monitor cell for updates. Do not launch a second worker while this PID is alive.


In [17]:
# Read-only progress monitor. Safe to rerun; it never starts/stops the worker.
if STATE_PATH.is_file():
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    print(json.dumps(state, indent=2))
else:
    print("Worker state has not been written yet. Check worker logs.")
if PID_PATH.is_file():
    try:
        pid = int(PID_PATH.read_text(encoding="utf-8").strip())
        print("Worker PID:", pid)
        print("Worker process alive:", is_pid_running(pid))
    except Exception as exc:
        print("Could not check worker PID:", repr(exc))
if (LOG_DIR / "rgb_worker_stderr.log").is_file():
    err_tail = (LOG_DIR / "rgb_worker_stderr.log").read_text(encoding="utf-8", errors="replace")[-3000:]
    if err_tail.strip():
        print("Latest stderr tail:\n", err_tail)


{
  "status": "completed",
  "device": "cuda:0",
  "gpu_name": "NVIDIA RTX A4000",
  "total_videos": 9654,
  "completed": 9654,
  "success": 9654,
  "skipped_existing": 0,
  "failed": 0,
  "elapsed_seconds": 4210.2,
  "report_path": "C:\\Users\\admin\\ViTPose_Action_Recognition\\Notebook_2_Improved_v2\\data\\rgb_features_r3d18_v2\\rgb_feature_extraction_report.csv"
}
Worker PID: 668
Worker process alive: False
Latest stderr tail:
 MB/s]
 45%|####4     | 57.2M/127M [00:05<00:06, 11.9MB/s]
 46%|####5     | 58.5M/127M [00:05<00:06, 11.9MB/s]
 47%|####6     | 59.8M/127M [00:05<00:05, 11.9MB/s]
 48%|####7     | 61.0M/127M [00:05<00:05, 11.9MB/s]
 49%|####8     | 62.2M/127M [00:05<00:05, 11.8MB/s]
 50%|####9     | 63.5M/127M [00:05<00:05, 11.8MB/s]
 51%|#####     | 64.8M/127M [00:05<00:05, 11.8MB/s]
 52%|#####1    | 66.0M/127M [00:05<00:05, 11.8MB/s]
 53%|#####2    | 67.2M/127M [00:05<00:05, 11.8MB/s]
 54%|#####3    | 68.5M/127M [00:06<00:05, 11.8MB/s]
 55%|#####4    | 69.8M/127M [00:06<00:0

In [18]:
# Final feature coverage audit. Run after the worker reports completion.
result_files = sorted(PER_VIDEO_DIR.glob("result_*.json"))
result_rows = []
for p in result_files:
    try:
        result_rows.append(json.loads(p.read_text(encoding="utf-8")))
    except Exception as exc:
        print("Unreadable result record:", p.name, repr(exc))
results_df = pd.DataFrame(result_rows)

expected_ids = set(selected["video_id"].astype(int))
valid_ids = set()
for row in selected.to_dict("records"):
    key = str(Path(row["video_path"]).resolve()).casefold()
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]
    feature_path = RGB_DIR / f"video_{int(row['video_id']):06d}_{digest}.npz"
    try:
        with np.load(feature_path, allow_pickle=False) as d:
            ok = (d["feature"].shape == (512,) and int(d["video_id"]) == int(row["video_id"])
                  and str(d["video_path"].item()) == str(row["video_path"]))
        if ok:
            valid_ids.add(int(row["video_id"]))
    except Exception:
        pass

missing_ids = sorted(expected_ids - valid_ids)
print("Expected source videos needed by sequences:", len(expected_ids))
print("Valid RGB feature files:", len(valid_ids))
print("Missing RGB features:", len(missing_ids))
if not results_df.empty and "status" in results_df:
    print("Per-video result statuses:")
    print(results_df["status"].value_counts(dropna=False).to_string())
if missing_ids:
    print("Examples still missing:", missing_ids[:20])
    print("Do not run Notebook 07 until missing RGB features are resolved.")
else:
    print("RGB feature coverage is complete for every source video used by train/val/test sequences.")
if not results_df.empty:
    results_df.to_csv(REPORT_PATH, index=False)
    print("Consolidated report:", REPORT_PATH)


Expected source videos needed by sequences: 9654
Valid RGB feature files: 9654
Missing RGB features: 0
Per-video result statuses:
status
success    9654
RGB feature coverage is complete for every source video used by train/val/test sequences.
Consolidated report: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\rgb_features_r3d18_v2\rgb_feature_extraction_report.csv


## Run order and safety notes

1. Finish Notebook 05 training first, or ensure no other process is using the GPU. Notebook 06 also requires GPU memory.
2. Run Cells 1–2 to verify the full sequence dataset and write the worker/configuration. Cell 1 explicitly checks the existing Anaconda interpreter and CUDA; do not continue if this check fails.
3. Run Cell 3 once to launch the detached worker with `C:\Users\admin\anaconda3\python.exe` (not the Jupyter kernel's Python 3.13). It refuses to launch a duplicate if its recorded PID is alive.
4. Use Cell 4 to monitor progress. It is safe to rerun; it does not start or stop extraction.
5. Run Cell 5 after completion to verify feature coverage.
6. Run Notebook 07 only when Cell 5 reports **zero missing RGB features**.

The worker writes one `.npz` feature file per source video into `data/rgb_features_r3d18_v2`. Each file contains a 512-D feature plus `video_id`, `video_path`, `label`, and `split`, matching Notebook 07's loader. It also saves per-video JSON results and a progress-state JSON. If interrupted, rerun Cells 1–3; valid feature files are skipped. Do not start a second worker while the recorded PID is alive.

This is frozen pretrained R3D-18 feature extraction, not end-to-end RGB model training. RGB + pose fusion training happens in Notebook 07.